In [ ]:
import os
import json
import pandas as pd

def load_finetuning_data(base_path):
	"""
	Reads all JSON files under base_path except logical_fallacy.json
	and merges them into a single pandas DataFrame with predefined columns.
	"""

	# Check path exists
	if not os.path.exists(base_path):
		raise FileNotFoundError(f"❌ 경로가 존재하지 않습니다: {base_path}")

	exclude_files = ["logical_fallacy.json", "correct.json", "training_data.json", "training_data_input_output_format.jsonl"]
	columns = ["question", "retrieved_passages", "previous_steps", "current_step", "error_type", "feedback"]
	data_rows = []

	for filename in os.listdir(base_path):
		if filename.endswith(".json") and filename not in exclude_files:
			file_path = os.path.join(base_path, filename)

			try:
				with open(file_path, "r", encoding="utf-8") as f:
					data = json.load(f)
					
					print(f"✅ 파일 로드 완료: {filename} (항목 수: {len(data) if isinstance(data, list) else 1})")

					if filename=="correct_paraphrased.json":
						# data에서 2500개만 랜덤 샘플링
						data = pd.DataFrame(data).sample(n=2500, random_state=42).to_dict(orient='records')

					# If JSON is a list of objects
					if isinstance(data, list):
						for entry in data:
							row = {col: entry.get(col, None) for col in columns}
							data_rows.append(row)

					# If JSON is a single object
					elif isinstance(data, dict):
						row = {col: data.get(col, None) for col in columns}
						data_rows.append(row)

			except json.JSONDecodeError:
				print(f"⚠️ JSON 형식 오류로 스킵됨: {filename}")
			except Exception as e:
				print(f"⚠️ 오류 발생 ({filename}): {e}")

	# Convert to DataFrame
	df = pd.DataFrame(data_rows, columns=columns)
	return df


# ------------------- 사용 예시 -------------------
base_path = "/workspace/daeyong/second_finetuning_data"
df = load_finetuning_data(base_path)

print("📌 총 데이터 개수:", len(df))
df

In [ ]:
from collections import Counter

print(Counter(df['error_type']))

In [ ]:
correct_df = pd.read_json("/workspace/daeyong/feedback/correct_feedback_morethan4_paraphrased.json")
print("📌 Correct feedback 데이터 개수:", len(correct_df))
correct_df

In [ ]:
final_df = pd.concat([df, correct_df], ignore_index=True)
del final_df['ideal_steps']
final_df

In [ ]:
Counter(final_df['error_type'])

In [ ]:
final_df.to_json("/workspace/daeyong/third_finetuning_data/training_data_paraphrased.json", orient="records", force_ascii=False, indent=2, lines=False)